# SpaNCy-Shift Stage 2 — neighbour-mean CONFIRMATION run (seeds 3–5)

**Why:** in `spancy_shift_nbrmean.ipynb` (seeds 0–2) the `nbr` arm (encoder input = own markers +
mean of the 15 spatial neighbours **within the same sample and scene**) beat the published model
on kBET at every seed (+0.015 / +0.022 / +0.024, mean 0.730 vs 0.710) with equal silhouette, but
failed the pre-set shape rule by one marker at one seed (CD31 peak 0.798 at seed 1). NOTCH1 var
also sat at 1.15–1.18 (cutoff 1.25). This run decides it on **new seeds** with the rule fixed in
advance.

**Arms per seed (3, 4, 5):** `full` (published config) and `nbr`, same init per seed, plus two
derived controls: **`full-noise`** / **`nbr-noise`** = Stage 1 + Gaussian noise with SD matched per
batch and marker to that arm's own correction (as in `spancy_shift_noisectrl_seeds.ipynb`).
`nbr`'s correction is ~8% larger than `full`'s, and kBET rewards perturbation size, so
`size effect = kBET(nbr-noise) − kBET(full-noise)` estimates how much of the gain is just size;
`structured gain = (nbr − full) − size effect`.

**Pre-set rule — `nbr` is ADOPTED only if ALL hold at seeds 3, 4 AND 5 (paired vs `full`):**
1. kBET higher every seed
2. silhouette ≥ `full` − 0.005 every seed
3. mean number of shape-distorted markers not higher than `full` (peak < 0.80 or var outside
   [0.80, 1.25], all markers)
4. bimodal markers untouched (`bim_delta` ≤ 0.001), no divergence
5. structured gain > 0 every seed

No re-reading of borderline cases afterwards: whatever §9 prints is the decision. Seeds 0–2 are
printed alongside (hard-coded) for the pooled 6-seed numbers only.

**To run:** Runtime → Run all (GPU). Upload **`spancy_shift_nbrmean.py`** (the FIXED version with
per-sample neighbours: its `nbr` training log says `Building spatial k-NN ... for ~75 scenes`).
6 trainings × ~8 min ≈ 50 min + kBET on 12 layers. **Paste back §9.**

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift_nbrmean.py (requested only if missing).
from google.colab import files
import os

NEEDED = ['spancy_shift_nbrmean.py', 'pospop_uniform.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')
src = open('spancy_shift_nbrmean.py').read()
assert 'per_sample=True' in src, 'OLD spancy_shift_nbrmean.py — upload the fixed one (per-sample neighbours)'
print('Module has per-sample neighbour fix: OK')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import iqr as _iqr
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift_nbrmean as ss
from spancy_shift_nbrmean import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    positive_population_table,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 3. Train `full` and `nbr` (seeds 3–5) + matched-noise controls

Same training call as the seeds 0–2 run. Each control is matched to its own arm's correction
(log1p, SD per batch and marker), with a fixed RNG per seed.

In [ ]:
SEEDS = [3, 4, 5]
KBET_SEEDS = SEEDS
POP_SEEDS = [3]
K_NEIGHBORS = 15

BASE = dict(
    n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512, k_neighbors=K_NEIGHBORS,
    hybrid_alpha=0.6, temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
    mmd_samples=256, grl_max=1.0, use_nbr_mean=False, w_mnn=0.0, mnn_k=5,
)
ARMS = {'full': {}, 'nbr': {'use_nbr_mean': True}}
KEY = {'full': 'full', 'nbr': 'nbr', 'full-noise': 'fullnoise', 'nbr-noise': 'nbrnoise'}


def lkey(arm, seed):
    return f'{KEY[arm]}_s{seed}'


def matched_noise(delta, rng):
    # Stage 1 + zero-mean Gaussian noise, SD matched per batch and marker to `delta` (log1p)
    g = base_log.copy()
    for b in np.unique(batches_arr):
        ii = np.where(batches_arr == b)[0]
        g[ii] += rng.standard_normal((len(ii), delta.shape[1])).astype(np.float32) * delta[ii].std(axis=0)
    return np.clip(np.expm1(g), 0, None).astype(np.float32)


adata_norm = None
train_info = {}
t0 = time.time()
for seed in SEEDS:
    for arm, over in ARMS.items():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        t_arm = time.time()
        m, sc, ref, hist = ss.train(
            adata, device_str=DEVICE,
            bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
            ref_sample_per_marker=ref_sample_per_marker,
            **{**BASE, **over},
        )
        train_min = (time.time() - t_arm) / 60
        torch.manual_seed(100)
        torch.cuda.manual_seed_all(100)
        out = ss.normalize_adata(
            adata, m, sc, ref, hybrid_alpha=BASE['hybrid_alpha'], k_neighbors=K_NEIGHBORS,
            inference_batch_size=2048, device_str=DEVICE,
            layer_name='normalized', keep_base_layer=True,
        )
        g = np.asarray(out.layers['normalized'], dtype=np.float32)
        if adata_norm is None:
            adata_norm = out
            del adata_norm.layers['normalized']
            X_base = np.asarray(adata_norm.layers['normalized_base'])
            base_log = np.log1p(np.clip(X_base, 0, None)).astype(np.float32)
            batches_arr = adata_norm.obs['batch_id'].astype(str).values
        else:
            del out
        adata_norm.layers[lkey(arm, seed)] = g
        delta = np.log1p(np.clip(g, 0, None)) - base_log
        noise_arm = f'{arm}-noise'
        adata_norm.layers[lkey(noise_arm, seed)] = matched_noise(
            delta, np.random.default_rng(1000 * seed + (0 if arm == 'full' else 1)))
        train_info[lkey(arm, seed)] = dict(train_min=train_min, mnn_pairs=0.0)
        train_info[lkey(noise_arm, seed)] = dict(train_min=0.0, mnn_pairs=0.0)
        del g, m, sc, ref, hist, delta
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'  seed {seed}  {arm:5s} done  train {train_min:.1f} min  (total {(time.time() - t0) / 60:.1f} min)')

ALL_ARMS = ['full', 'nbr', 'full-noise', 'nbr-noise']
LAYERS = [('normalized_base', 'Stage 1', None, None)]
LAYERS += [(lkey(a, s), f'{a} s{s}', a, s) for s in SEEDS for a in ALL_ARMS]
print(f'\n{len(LAYERS)} layers: {[l[1] for l in LAYERS]}')

In [ ]:
# 3b. Divergence check — run after §3, before §4.
# Without the Huber term (no-Huber, all-off) nothing bounds the size of the correction, and the
# expm1 at inference overflows (max 4e10 / inf). Such layers break the metric cells (inf -> histogram
# and silhouette errors), so they are reported here and dropped from LAYERS: their verdict is
# "diverged". Criterion: any non-finite value, or > 0.1% of entries above 1.2x the raw per-marker
# log1p max (the same 1.2x cap rule as SpaNCy-GNN).
CAP = 1.2
raw_max = np.log1p(np.clip(X_raw, 0, None).max(axis=0))
diverged, div_rows = {}, []
for key, label, arm, seed in LAYERS:
    X = np.asarray(adata_norm.layers[key])
    with np.errstate(over='ignore', invalid='ignore'):
        X_log = np.log1p(np.clip(X, 0, None))
    finite = np.isfinite(X_log)
    over = ~finite | (X_log > CAP * raw_max)
    frac_over = over.mean(axis=0)
    bad_markers = [f'{marker_names[i]} {100 * frac_over[i]:.1f}%'
                   for i in np.argsort(-frac_over) if frac_over[i] > 0.001]
    is_div = (~finite).any() or over.mean() > 0.001
    if is_div and arm is not None:
        diverged[key] = (arm, seed)
    div_rows.append(dict(layer=label, nonfinite=int((~finite).sum()),
                         pct_over_cap=round(100 * float(over.mean()), 3),
                         max_log1p=round(float(X_log[finite].max()), 2),
                         diverged=is_div, markers_over_cap=', '.join(bad_markers[:6])))
    del X, X_log, finite, over
    gc.collect()

print(f'Raw log1p max over markers: {raw_max.max():.2f}  (cap = {CAP}x per marker)')
print(pd.DataFrame(div_rows).to_string(index=False))
LAYERS = [l for l in LAYERS if l[0] not in diverged]
print(f'\nDropped {len(diverged)} diverged layers from the metric cells: '
      f'{sorted(f"{a} s{s}" for a, s in diverged.values())}')

## 3c. Histograms

Per-sample overlay curves (one line per sample, tab20) in log1p space, one row per marker:
Raw | Stage 1 | full | nbr | full-noise | nbr-noise, for `HIST_SEED`. Bin edges come from **raw**
per marker (as in the article figures); the y-limit is the max over all columns so that
compression (taller, narrower curves) is not clipped. Inline: the markers whose shape `nbr`
changed most in seeds 0–2 (NOTCH1 broader, CD31 flatter peak, aSMA / DAPI_R1 narrower). Full PDF
(all 20 markers, every seed as a separate file) in `histograms_nbr_confirm/`.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import os

HIST_DIR = 'histograms_nbr_confirm'
HIST_SEED = SEEDS[0]
WATCH = ['NOTCH1', 'CD31', 'aSMA', 'DAPI_R1']
N_HIST_BINS = 100
MARKERS_PER_PAGE = 4
RAW = '__RAW__'
os.makedirs(HIST_DIR, exist_ok=True)
M_h = len(marker_names)

sample_ids_h = adata_norm.obs['sample_id'].values
samples_h = sorted(np.unique(sample_ids_h).tolist())
sidx_h = {s: np.where(sample_ids_h == s)[0] for s in samples_h}
colors_h = plt.cm.tab20(np.linspace(0, 1, len(samples_h)))
X_raw_log_h = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)

edges_h = {}
for m in range(M_h):
    v = X_raw_log_h[:, m]
    hi = float(np.percentile(v, 99.9))
    edges_h[m] = np.linspace(float(v.min()), hi if hi > v.min() else float(v.max()), N_HIST_BINS + 1)


def _hist_layers(seed):
    return [(RAW, 'Raw'), ('normalized_base', 'Stage 1')] + [
        (lkey(a, seed), f'{a} s{seed}') for a in ['full', 'nbr', 'full-noise', 'nbr-noise']
        if lkey(a, seed) in adata_norm.layers]


def _densities(layers):
    dens = {}
    for key, _ in layers:
        XL = X_raw_log_h if key == RAW else np.log1p(
            np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
        for m in range(M_h):
            for s in samples_h:
                dens[(key, m, s)], _ = np.histogram(XL[sidx_h[s], m], bins=edges_h[m], density=True)
        del XL
        gc.collect()
    ymax = {m: max(d.max() for (k, mm, s), d in dens.items() if mm == m) for m in range(M_h)}
    return dens, ymax


def _plot_rows(marker_idx, layers, dens, ymax):
    fig, axes = plt.subplots(len(marker_idx), len(layers),
                             figsize=(2.9 * len(layers), 2.3 * len(marker_idx)),
                             squeeze=False, sharex='row')
    for r, m in enumerate(marker_idx):
        ctr = 0.5 * (edges_h[m][:-1] + edges_h[m][1:])
        for c, (key, label) in enumerate(layers):
            ax = axes[r, c]
            for s, col in zip(samples_h, colors_h):
                ax.plot(ctr, dens[(key, m, s)], color=col, lw=0.8)
            ax.set_ylim(0, ymax[m] * 1.05)
            if r == 0:
                ax.set_title(label, fontsize=9)
            if c == 0:
                ax.set_ylabel(marker_names[m], fontsize=9)
            ax.tick_params(labelsize=7)
    fig.tight_layout()
    return fig


for seed in SEEDS:
    layers_h = _hist_layers(seed)
    dens_h, ymax_h = _densities(layers_h)
    if seed == HIST_SEED:
        _plot_rows([marker_names.index(n) for n in WATCH if n in marker_names], layers_h, dens_h, ymax_h)
        plt.show()
    pdf_path = os.path.join(HIST_DIR, f'nbr_confirm_histograms_s{seed}.pdf')
    with PdfPages(pdf_path) as pdf:
        for p in range(0, M_h, MARKERS_PER_PAGE):
            fig = _plot_rows(list(range(p, min(p + MARKERS_PER_PAGE, M_h))), layers_h, dens_h, ymax_h)
            pdf.savefig(fig, dpi=150)
            plt.close(fig)
    print(f'Saved {pdf_path}')
    del dens_h
    gc.collect()

try:
    import shutil
    from google.colab import files as _files
    shutil.make_archive(HIST_DIR, 'zip', HIST_DIR)
    _files.download(HIST_DIR + '.zip')
except Exception as e:
    print(f'(download skipped: {e})')

## 4. 1D shape + correction on bimodal markers

Same per-(marker, sample) ratios as cell 6b of the other notebooks (`_shape_ratios` copied verbatim),
but computed for **all** markers so the bimodal ones (ECAD, …) are visible. `bim_delta` = mean
|log1p(arm) − log1p(Stage 1)| over bimodal markers: ~0 means bimodal markers were left alone.

In [ ]:
N_SHAPE_BINS = 80
MIN_CELLS = 50
PEAK_LO, VAR_LO, VAR_HI = 0.80, 0.80, 1.25

sample_ids_shape = adata_norm.obs['sample_id'].values
sample_idx = {s: np.where(sample_ids_shape == s)[0] for s in np.unique(sample_ids_shape)}
sample_idx = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


shape_rows, shape_sum = [], {}
for key, label, arm, seed in LAYERS:
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        shape_rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame([r for r in shape_rows if r['key'] == key]).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )
    del X_log
    gc.collect()
shape_df = pd.DataFrame(shape_rows)

print('Shape summary (peak/var/iqr = unimodal means; dist = ALL markers incl. bimodal):')
print(pd.DataFrame(shape_sum).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 5. Silhouette (all seeds)

In [ ]:
# Silhouette — UniFORM-style, 3 cell types, PER SAMPLE (same method + guard as the other notebooks).
N_SIL = 3000
MIN_TYPE_CELLS = 50

anchor_thresholds = {}
for mname in ['ECAD', 'CD45', 'aSMA']:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(X_log_preview[:, mi].reshape(-1, 1))
    anchor_thresholds[mname] = float(np.mean(gmm.means_))
print('Anchor GMM thresholds (log1p, raw):', {k: round(v, 3) for k, v in anchor_thresholds.items()})

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']] = 2
cell_type[X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']] = 1
cell_type[X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']] = 0

# Per-sample selection is decided once (deterministic guard, rng 42 per sample as before).
sample_ids_all = adata_norm.obs['sample_id'].values
sil_sel, dropped = {}, []
for sample in sorted(np.unique(sample_ids_all)):
    smask_idx = np.where(sample_ids_all == sample)[0]
    s_labels = cell_type[smask_idx]
    big = [c for c in (0, 1, 2) if int((s_labels == c).sum()) >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sil_sel[sample] = (smask_idx[sel], s_labels[sel])

sil_rows = {}
for key in ['raw'] + [k for k, _, _, _ in LAYERS]:
    X = X_raw if key == 'raw' else np.asarray(adata_norm.layers[key])
    sil_rows[key] = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
                     for s, (ii, lab) in sil_sel.items()}
sil_df = pd.DataFrame(sil_rows)
sil_df.loc['Mean'] = sil_df.mean()
print(f'{len(sil_sel)} samples passed the guard (dropped: {dropped})')
print(sil_df.loc[['Mean']].round(4).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).to_string())

## 6. Positive population (`POP_SEEDS`)

In [ ]:
pop_sum = {}
for key, label, arm, seed in LAYERS:
    if seed is not None and seed not in POP_SEEDS:
        continue
    pt = positive_population_table(adata_norm, norm_layer=key, sample_col='sample_id')
    s = pt.groupby('marker')['delta'].mean()
    pop_sum[key] = dict(pass_5pct=int((s.abs() < 5).sum()),
                        mean_abs_delta=float(pt['delta'].abs().mean()))
    print(f'  {label:16s} {pop_sum[key]["pass_5pct"]}/20 within +/-5%   '
          f'mean |delta| {pop_sum[key]["mean_abs_delta"]:.2f}%')

## 6b. Positive population — UniFORM definition

`pospop_uniform.py`: local GMM threshold per RAW sample vs ONE global GMM threshold on all
NORMALIZED cells pooled, per marker (UniFORM STAR Methods); GMM threshold copied from UniFORM's
code (natural log, 2 components, max of the negative class). Change = percentage points, mean ± SD
over samples (our reading — UniFORM's evaluation script is not published). Reported next to the
current metric (§6, raw threshold transferred to normalized), which answers a different question.
Local thresholds depend on raw data only and are computed once.

In [ ]:
import pospop_uniform as pu

# Set these from the BEST MATCH printed by mxnorm_benchmark_pospop_uniform.ipynb §5x:
#   'A plot_gmm (zeros->log 0)' -> POPU_DROP_BELOW_ONE = False
#   'B preprocessing (drop <1)' -> POPU_DROP_BELOW_ONE = True
#   aggregation 'mean_abs' (mean of |change|) or 'abs_mean' (|mean change|) -> POPU_AGG
POPU_DROP_BELOW_ONE = False
POPU_AGG = 'mean_abs'
MAX_FIT_CELLS = 300_000
PKW = dict(drop_below_one=POPU_DROP_BELOW_ONE, max_cells=MAX_FIT_CELLS)
AGG_COL = {'mean_abs': 'mean_abs_delta_pp', 'abs_mean': 'abs_mean_pp'}[POPU_AGG]
print(f'Definition: drop_below_one={POPU_DROP_BELOW_ONE}, aggregation={POPU_AGG}')

t0 = time.time()
sample_ids_u = adata_norm.obs['sample_id'].astype(str).values
local_thr_u = pu.local_thresholds(X_raw, sample_ids_u, marker_names, **PKW)
print(f'Local raw thresholds: {len(local_thr_u)} (marker, sample) pairs, {time.time() - t0:.0f} s')

POPU_LAYERS = [('normalized_base', 'Stage 1')] + [
    (lkey(a, s), f'{a} s{s}') for s in SEEDS for a in ['full', 'nbr']]
popu_sum, popu_tab = {}, {}
for key, label in POPU_LAYERS:
    t1 = time.time()
    tab, g_thr = pu.positive_population_uniform(
        X_raw, np.asarray(adata_norm.layers[key]), sample_ids_u, marker_names,
        local_thr=local_thr_u, **PKW)
    s = pu.summarize_uniform(tab)
    s['metric'] = s[AGG_COL]
    s['pass'] = s['metric'] < 5
    popu_tab[key] = s
    popu_sum[key] = dict(pass_5pp=int(s['pass'].sum()), mean_abs_pp=float(s['metric'].mean()),
                         mean_sd_pp=float(s.sd_delta_pp.mean()))
    print(f'  {label:12s} {popu_sum[key]["pass_5pp"]:2d}/20 within +/-5 pp   '
          f'mean |change| {popu_sum[key]["mean_abs_pp"]:.2f} pp   mean SD {popu_sum[key]["mean_sd_pp"]:.2f} pp'
          f'   ({time.time() - t1:.0f} s)')

print('\nPer marker, mean change (pp) +/- SD over samples:')
first = lkey('full', SEEDS[0]), lkey('nbr', SEEDS[0])
cmp_u = pd.DataFrame({
    'Stage 1': popu_tab['normalized_base'].metric,
    'S1 sd': popu_tab['normalized_base'].sd_delta_pp,
    f'full s{SEEDS[0]}': popu_tab[first[0]].metric,
    f'nbr s{SEEDS[0]}': popu_tab[first[1]].metric,
    f'nbr sd': popu_tab[first[1]].sd_delta_pp,
})
print(cmp_u.round(2).to_string())

print('\nSide by side with the current metric (§6, seed POP_SEEDS only):')
for key, label in POPU_LAYERS:
    cur = pop_sum.get(key)
    cur_txt = f'{cur["pass_5pct"]}/20' if cur else '  - '
    print(f'  {label:12s} current {cur_txt:>6s}   UniFORM-style {popu_sum[key]["pass_5pp"]}/20')

d_nbr = [popu_sum[lkey('nbr', s)]['mean_abs_pp'] - popu_sum[lkey('full', s)]['mean_abs_pp'] for s in SEEDS]
print(f'\nnbr - full, mean |change| (pp) per seed: {np.round(d_nbr, 3).tolist()}  (negative = nbr closer to raw gating)')

In [ ]:
# Optional: export layers for mxnorm_benchmark_pospop_uniform.ipynb (same float32 .npy format as
# the spancy_shift_explore export; no Drive). Set EXPORT = True to run.
EXPORT = False
EXPORT_KEYS = [lkey('nbr', SEEDS[0]), lkey('full', SEEDS[0])]
if EXPORT:
    import json as _json
    os.makedirs('/content/spancy_layers', exist_ok=True)
    np.save('/content/spancy_layers/confirm_obs_names.npy', np.asarray(adata_norm.obs_names).astype(str))
    for k in EXPORT_KEYS:
        p = f'/content/spancy_layers/{k}_normalized.npy'
        np.save(p, np.asarray(adata_norm.layers[k], dtype=np.float32))
        print(f'saved {p}')
    with open('/content/spancy_layers/confirm_meta.json', 'w') as f:
        _json.dump(dict(keys=EXPORT_KEYS, shape=list(adata_norm.shape),
                        note='spancy_shift_nbr_confirm.ipynb, published config + nbr flag'), f)
    from google.colab import files as _files
    for k in EXPORT_KEYS:
        _files.download(f'/content/spancy_layers/{k}_normalized.npy')
    _files.download('/content/spancy_layers/confirm_obs_names.npy')

## 8. kBET (`KBET_SEEDS`)

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

# Stage 1 kBET is deterministic (0.6202) -> skipped.
ABL_LABEL = {k: l for k, l, _, _ in LAYERS}
EVAL_LAYERS = [k for k, _, _, s in LAYERS if s is not None and s in KBET_SEEDS]
print(f'kBET on {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

## 9. Confirmation result (paste this)

Decision = the pre-set rule in the title cell, on seeds 3–5 only. Seeds 0–2 (from
`spancy_shift_nbrmean.ipynb`) are shown for the pooled 6-seed summary.

In [ ]:
recs = []
for key, label, arm, seed in LAYERS:
    if arm is None:
        continue
    r = dict(arm=arm, seed=seed, sil=float(sil_df.loc['Mean', key]),
             **{k: v for k, v in shape_sum[key].items() if k != 'dist_list'})
    if key in all_kbet:
        kv = [x['kBET'] for x in all_kbet[key].values() if not np.isnan(x['kBET'])]
        r['kBET'] = float(np.mean(kv)) if kv else np.nan
    if key in pop_sum:
        r['pos_pop'] = pop_sum[key]['pass_5pct']
    recs.append(r)
res = pd.DataFrame(recs)

cols = [c for c in ['kBET', 'sil', 'peak', 'var', 'iqr', 'dist', 'bim_delta', 'uni_delta'] if c in res]
print('=' * 100)
print(f'SEEDS {SEEDS}: MEAN +/- SD')
print('=' * 100)
print(res.groupby('arm', sort=False)[cols].agg(['mean', 'std']).round(4).to_string())
print('\nPer seed:')
print(res[['arm', 'seed'] + [c for c in ['kBET', 'sil', 'dist', 'uni_delta', 'pos_pop'] if c in res]]
      .sort_values(['seed', 'arm']).round(4).to_string(index=False))

print('\nPer-group kBET:')
for key, label, arm, seed in LAYERS:
    if key in all_kbet:
        print(f'  {label:16s} ' + '  '.join(f'{g} {v["kBET"]:.3f}' for g, v in all_kbet[key].items()))

print('\nDistorted markers (all markers; var of NOTCH1 / CD31 shown — the two to watch):')
for key, label, arm, seed in LAYERS:
    if arm in ('full', 'nbr'):
        d = shape_df[shape_df.key == key].set_index('marker')
        print(f'  {label:10s} {shape_sum[key]["dist"]}  {shape_sum[key]["dist_list"]:24s} '
              f'NOTCH1 var {d.loc["NOTCH1", "var"]:.3f}  CD31 peak {d.loc["CD31", "peak"]:.3f} '
              f'iqr {d.loc["CD31", "iqr"]:.3f}')

P = res.set_index(['arm', 'seed'])
dk = pd.Series({s: P.loc[('nbr', s), 'kBET'] - P.loc[('full', s), 'kBET'] for s in SEEDS})
dsize = pd.Series({s: P.loc[('nbr-noise', s), 'kBET'] - P.loc[('full-noise', s), 'kBET'] for s in SEEDS})
dstruct = dk - dsize
ds = pd.Series({s: P.loc[('nbr', s), 'sil'] - P.loc[('full', s), 'sil'] for s in SEEDS})
ddist = float(np.mean([P.loc[('nbr', s), 'dist'] - P.loc[('full', s), 'dist'] for s in SEEDS]))
bim = float(max(P.loc[('nbr', s), 'bim_delta'] for s in SEEDS))
n_div = sum(1 for k in diverged if k.startswith('nbr_s') or k.startswith('full_s'))

print('\n' + '=' * 100)
print('PAIRED nbr - full (seeds 3-5)')
print('=' * 100)
for name, sr in [('kBET', dk), ('size effect (noise)', dsize), ('structured gain', dstruct), ('silhouette', ds)]:
    print(f'  {name:20s} ' + '  '.join(f's{s} {v:+.4f}' for s, v in sr.items())
          + f'   mean {sr.mean():+.4f} +/- {sr.std():.4f}')
print(f'  distorted markers: mean {ddist:+.2f}   bim_delta max {bim:.4f}   diverged layers {n_div}')

checks = {
    '1 kBET higher every seed': bool((dk > 0).all()),
    '2 silhouette >= full - 0.005 every seed': bool((ds >= -0.005).all()),
    '3 no extra distorted markers (mean)': ddist <= 0,
    '4 bimodal untouched, no divergence': bim <= 0.001 and n_div == 0,
    '5 structured gain > 0 every seed': bool((dstruct > 0).all()),
}
for name, ok in checks.items():
    print(f'  [{"x" if ok else " "}] {name}')
print(f'\nVERDICT: {"ADOPT nbr" if all(checks.values()) else "REJECT nbr (report as a lead)"}')

# Pooled 6 seeds (0-2 hard-coded from spancy_shift_nbrmean.ipynb, 2026-10-06)
PREV = {'full': dict(kBET=[0.7115, 0.6996, 0.7178], sil=[0.3581, 0.3595, 0.3648], dist=[0, 0, 0]),
        'nbr':  dict(kBET=[0.7269, 0.7214, 0.7415], sil=[0.3585, 0.3616, 0.3623], dist=[0, 1, 0])}
print('\n' + '=' * 100)
print('POOLED seeds 0-5 (descriptive only; the decision above uses seeds 3-5)')
print('=' * 100)
for arm in ['full', 'nbr']:
    k = PREV[arm]['kBET'] + [P.loc[(arm, s), 'kBET'] for s in SEEDS]
    si = PREV[arm]['sil'] + [P.loc[(arm, s), 'sil'] for s in SEEDS]
    di = PREV[arm]['dist'] + [P.loc[(arm, s), 'dist'] for s in SEEDS]
    print(f'  {arm:5s} kBET {np.mean(k):.4f} +/- {np.std(k, ddof=1):.4f}   '
          f'sil {np.mean(si):.4f} +/- {np.std(si, ddof=1):.4f}   distorted/seed {np.mean(di):.2f}')
dk6 = np.array(PREV['nbr']['kBET']) - np.array(PREV['full']['kBET'])
dk6 = np.concatenate([dk6, dk.values])
print(f'  paired kBET nbr - full over 6 seeds: {np.round(dk6, 4).tolist()}  '
      f'mean {dk6.mean():+.4f} +/- {dk6.std(ddof=1):.4f}, positive {int((dk6 > 0).sum())}/6')